# latency profile

this notebook reads structured logs from a run, extracts per-stage timings, and plots them as a stacked bar chart per turn. the structured log key is `turn.metrics`.

stages tracked:
- `stt_done - stt_start` -> STT
- `rag_done - rag_start` -> retrieval
- `llm_first_token - rag_done` -> LLM TTFT
- `tts_first_byte - llm_first_token` -> TTS first chunk
- `done - tts_first_byte` -> remaining TTS playout

In [ ]:
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

LOG = Path("../var/agent.log")

In [ ]:
rows = []
for line in LOG.read_text().splitlines():
    try:
        m = json.loads(line)
    except Exception:
        continue
    if m.get('msg') == 'turn.metrics':
        rows.append(m)
df = pd.DataFrame(rows)
df.head()

In [ ]:
df['stt'] = df['stt_done_ms'] - df['stt_start_ms']
df['rag'] = df['rag_done_ms'] - df['rag_start_ms']
df['llm_ttft'] = df['llm_first_token_ms'] - df['rag_done_ms']
df['tts_first'] = df['tts_first_byte_ms'] - df['llm_first_token_ms']
df['tts_rest'] = df['done_ms'] - df['tts_first_byte_ms']
df[['stt', 'rag', 'llm_ttft', 'tts_first', 'tts_rest']].describe()

In [ ]:
ax = df[['stt','rag','llm_ttft','tts_first','tts_rest']].head(40).plot(kind='bar', stacked=True, figsize=(10,4))
ax.set_ylabel('ms')
ax.set_xlabel('turn')
ax.set_title('per-turn latency stack')
plt.tight_layout()
plt.show()